In [0]:
# 08_publish_site_to_github.ipynb
import os
import time
import base64
import requests
import jwt

# GitHub repo target
repo_owner = "KingBain"
repo_name = "fsdh-artemis-demo"
branch_name = "main"

# Databricks secret scope
secret_scope = "github-app"

# Local generated site root
site_root = "/Workspace/Users/john.bain@ssc-spc.gc.ca/Artemis 2/fsdh-artemis-demo"

# Files to publish: local path -> repo path
files_to_publish = {
    os.path.join(site_root, "data", "mission-data.json"): "data/mission-data.json",
}

github_api_base = "https://api.github.com"

print("Publish target:")
print(f"- Repo: {repo_owner}/{repo_name}")
print(f"- Branch: {branch_name}")
print("Files to publish:")
for local_path, repo_path in files_to_publish.items():
    print(f"  {local_path} -> {repo_path}")

In [0]:
app_id = dbutils.secrets.get(scope=secret_scope, key="app-id")
private_key_pem = dbutils.secrets.get(scope=secret_scope, key="private-key")

print("GitHub App secrets loaded successfully.")

In [0]:
now = int(time.time())

jwt_payload = {
    "iat": now - 60,
    "exp": now + (9 * 60),
    "iss": app_id,
}

app_jwt = jwt.encode(jwt_payload, private_key_pem, algorithm="RS256")

app_headers = {
    "Authorization": f"Bearer {app_jwt}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

print("GitHub App JWT created successfully.")

In [0]:
installation_url = f"{github_api_base}/repos/{repo_owner}/{repo_name}/installation"
installation_resp = requests.get(installation_url, headers=app_headers, timeout=30)

print(f"Installation lookup status: {installation_resp.status_code}")
installation_resp.raise_for_status()

installation_id = installation_resp.json()["id"]
print(f"Resolved installation ID: {installation_id}")

access_token_url = f"{github_api_base}/app/installations/{installation_id}/access_tokens"
access_token_resp = requests.post(access_token_url, headers=app_headers, timeout=30)

print(f"Access token request status: {access_token_resp.status_code}")
access_token_resp.raise_for_status()

installation_token = access_token_resp.json()["token"]

repo_headers = {
    "Authorization": f"Bearer {installation_token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

print("Generated GitHub installation token.")

In [0]:
missing_files = []

for local_path in files_to_publish.keys():
    if not os.path.exists(local_path):
        missing_files.append(local_path)

if missing_files:
    raise FileNotFoundError(
        "Missing files to publish:\\n" + "\\n".join(missing_files)
    )

print("All local files exist and are ready to publish.")

In [0]:
def publish_file_to_github(local_path, repo_path):
    with open(local_path, "rb") as f:
        raw_bytes = f.read()

    content_b64 = base64.b64encode(raw_bytes).decode("utf-8")
    contents_url = f"{github_api_base}/repos/{repo_owner}/{repo_name}/contents/{repo_path}"

    get_resp = requests.get(
        contents_url,
        headers=repo_headers,
        params={"ref": branch_name},
        timeout=30,
    )

    existing_sha = None

    if get_resp.status_code == 200:
        existing_sha = get_resp.json()["sha"]
        print(f"Existing file found for {repo_path}")
    elif get_resp.status_code == 404:
        print(f"Creating new file: {repo_path}")
    else:
        raise Exception(
            f"Failed to read existing GitHub file {repo_path}: "
            f"{get_resp.status_code} {get_resp.text}"
        )

    payload = {
        "message": f"chore: update {repo_path}",
        "content": content_b64,
        "branch": branch_name,
    }

    if existing_sha:
        payload["sha"] = existing_sha

    put_resp = requests.put(
        contents_url,
        headers=repo_headers,
        json=payload,
        timeout=60,
    )

    print(f"Publish status for {repo_path}: {put_resp.status_code}")
    put_resp.raise_for_status()

    return put_resp.json()

In [0]:
publish_results = []

for local_path, repo_path in files_to_publish.items():
    print(f"Publishing {repo_path}...")
    result = publish_file_to_github(local_path, repo_path)
    publish_results.append({
        "repo_path": repo_path,
        "result": result
    })
    print(f"Published: {repo_path}")

In [0]:
print("Databricks artifact publish complete.")
print("Published files:")

for item in publish_results:
    print(f"- {item['repo_path']}")